In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv


In [2]:
import pandas as pd 
import torch 
import numpy as np
import torch.nn as nn
import random
from torch.utils.data import Dataset,DataLoader
import re
from collections import Counter
import gensim.downloader as api
random.seed(42)
torch.manual_seed(42)
device="cuda" if torch.cuda.is_available() else "cpu"
print(f"device : {device}")

device : cuda


In [3]:
glove = api.load("glove-wiki-gigaword-100")
emb_dim = glove.vector_size

[==================================================] 100.0% 128.1/128.1MB downloaded


In [4]:
df=pd.read_csv("/kaggle/input/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv")

In [5]:
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


In [6]:
df.sentiment.value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

In [7]:
df=df.sample(frac=1,random_state=42).reset_index()

In [8]:
train_size=int(0.7*len(df))
dev_size=int(0.15*len(df))
test_size=int(0.15*len(df))
train_set_reviews,train_set_sentiment=df.review.tolist()[:train_size],df.sentiment.tolist()[:train_size]
dev_set_reviews,dev_set_sentiment=df.review.tolist()[train_size:train_size+dev_size],df.sentiment.tolist()[train_size:train_size+dev_size]
test_set_reviews,test_set_sentiment=df.review.tolist()[train_size+dev_size:],df.sentiment.tolist()[train_size+dev_size:]
print(len(train_set_reviews),len(train_set_sentiment))
print(len(dev_set_reviews),len(dev_set_sentiment))
print(len(test_set_reviews),len(test_set_sentiment))

35000 35000
7500 7500
7500 7500


In [33]:
def tokenize(text):
    return re.findall(r"\w+", text.lower())

# Tokenize once, keep one token list per review
texts = df["review"].tolist()
tokenized = [tokenize(t) for t in texts]

# Count words and build vocab
counter = Counter(w for sent in tokenized for w in sent)
vocab = {"<pad>": 0, "<unk>": 1}
for word in counter:
    vocab[word] = len(vocab)

def encode(tokens, maxlen=265):
    ids = [vocab.get(t, vocab["<unk>"]) for t in tokens][:maxlen]
    ids = [vocab["<pad>"]] * (maxlen - len(ids))+ids
    return ids

maxlen = 500   
X = torch.tensor([encode(t, maxlen) for t in tokenized])
matrix =np.random.normal(scale=0.1,size=(len(vocab),emb_dim))
matrix[vocab["<pad>"]]=0
for word , id  in vocab.items():
    if word in glove:
        matrix[id]=glove[word]
    
matrix=torch.tensor(matrix,dtype=torch.float32)
embedding=nn.Embedding.from_pretrained(matrix, freeze=True, padding_idx=0)

In [34]:
X.shape

torch.Size([50000, 500])

In [35]:
class myData(Dataset):
    def __init__(self,reviews,sentiment):
        self.reviews=[embedding(torch.tensor((encode(tokenize(review))))) for review in reviews]
        self.sentiment=sentiment 
    def __len__(self):
        return len(self.reviews)
    def __getitem__(self,idx):
        return torch.tensor(self.reviews[idx]),torch.tensor(float(self.sentiment[idx]=="positive"))
        

In [36]:
train_data=myData(train_set_reviews,train_set_sentiment)
dev_data=myData(dev_set_reviews,dev_set_sentiment)
test_data=myData(test_set_reviews,test_set_sentiment)

In [37]:
train_loader=DataLoader(dataset=train_data,batch_size=64,shuffle=True)
dev_loader=DataLoader(dataset=dev_data,batch_size=64,shuffle=True)
test_loader=DataLoader(dataset=test_data,batch_size=64,shuffle=True)

In [38]:
next(iter(train_loader))

/tmp/ipykernel_49/48991422.py:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(self.reviews[idx]),torch.tensor(float(self.sentiment[idx]=="positive"))


[tensor([[[ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
          ...,
          [-0.0138,  0.3822,  0.5324,  ..., -1.0426,  0.2885,  0.6306],
          [ 0.9208, -0.3681, -0.0306,  ..., -0.7537,  0.3855,  0.0680],
          [ 0.3825,  0.1482,  0.6060,  ...,  0.0589,  0.0911,  0.4728]],
 
         [[ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
          ...,
          [-0.1529, -0.2428,  0.8984,  ..., -0.5910,  1.0039,  0.2066],
          [-0.6041,  0.6481,  0.4345,  ...,  0.1934, -0.1996,  0.6053],
          [ 0.1218, -0.0589,  0.8779,  ..., -0.3557,  0.7620, -0.1426]],
 
         [[ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
          [ 0.0000,  0.0000,

In [78]:
class Model(nn.Module):
    def __init__(self,input_size,hidden,n_classes=1):
        super().__init__()
        self.emb_drop = nn.Dropout(0.3) 
        self.lstm=nn.LSTM(input_size=input_size,     
                            hidden_size=hidden,    
                            num_layers=2,      
                            batch_first=True,  
                            dropout=0.3,      
                            bidirectional=True)
        self.drop = nn.Dropout(0.4)
        self.FC=nn.Linear(hidden,n_classes)
        
    def forward(self,x):
            x = self.emb_drop(x)
            _, (h_n, _) = self.lstm(x)
            return self.FC(self.drop(h_n[-1])).squeeze(1)
        
            
              

In [79]:
model=Model(emb_dim,128,1)
criterion=nn.BCEWithLogitsLoss()
optim=torch.optim.Adam(model.parameters(),lr=0.001)

In [80]:
def train(model, criterion, optim, n_epochs, Loader):
    model.to(device)
    model.train()
    n_batches = len(Loader)
    for epoch in range(n_epochs):
        total, n = 0, 0
        for i, (X, Y) in enumerate(Loader):
            X, Y = X.to(device), Y.to(device)

            optim.zero_grad()
            y_preds = model(X)
            loss = criterion(y_preds, Y)
            loss.backward()
            optim.step()

            total += loss.item()
            n += 1

            if (i + 1) % 50 == 0 or (i + 1) == n_batches:
                print(f"  epoch {epoch+1}/{n_epochs} | batch {i+1}/{n_batches} "
                      f"| loss {loss.item():.4f}")

        print(f"epoch {epoch+1}/{n_epochs} : Loss={total/n:.4f}")

In [81]:
train(model,criterion,optim,20,train_loader)

/tmp/ipykernel_49/48991422.py:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(self.reviews[idx]),torch.tensor(float(self.sentiment[idx]=="positive"))


  epoch 1/20 | batch 50/547 | loss 0.6926
  epoch 1/20 | batch 100/547 | loss 0.6849
  epoch 1/20 | batch 150/547 | loss 0.7014
  epoch 1/20 | batch 200/547 | loss 0.6803
  epoch 1/20 | batch 250/547 | loss 0.7012
  epoch 1/20 | batch 300/547 | loss 0.6937
  epoch 1/20 | batch 350/547 | loss 0.6914
  epoch 1/20 | batch 400/547 | loss 0.7014
  epoch 1/20 | batch 450/547 | loss 0.7069
  epoch 1/20 | batch 500/547 | loss 0.6897
  epoch 1/20 | batch 547/547 | loss 0.6918
epoch 1/20 : Loss=0.6930
  epoch 2/20 | batch 50/547 | loss 0.6914
  epoch 2/20 | batch 100/547 | loss 0.6913
  epoch 2/20 | batch 150/547 | loss 0.6809
  epoch 2/20 | batch 200/547 | loss 0.6767
  epoch 2/20 | batch 250/547 | loss 0.6953
  epoch 2/20 | batch 300/547 | loss 0.6913
  epoch 2/20 | batch 350/547 | loss 0.6897
  epoch 2/20 | batch 400/547 | loss 0.6912
  epoch 2/20 | batch 450/547 | loss 0.6843
  epoch 2/20 | batch 500/547 | loss 0.6934
  epoch 2/20 | batch 547/547 | loss 0.6663
epoch 2/20 : Loss=0.6892
  epoc

In [82]:
from sklearn.metrics import accuracy_score, recall_score, precision_score

def Evaluate(model, Loader):
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for X, Y in Loader:
            X, Y = X.to(device), Y.to(device)
            y_preds = model(X)                        
            probs = torch.sigmoid(y_preds)          
            preds = (probs > 0.5).long()            
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(Y.cpu().tolist())

    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds)
    rec = recall_score(all_labels, all_preds)

    print(f"Accuracy: {acc:.4f} | Precision: {prec:.4f} | Recall: {rec:.4f}")
    return acc, prec, rec

In [83]:
Evaluate(model, train_loader)

/tmp/ipykernel_49/48991422.py:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(self.reviews[idx]),torch.tensor(float(self.sentiment[idx]=="positive"))


Accuracy: 0.9459 | Precision: 0.9282 | Recall: 0.9663


(0.9458571428571428, 0.9282417582417583, 0.9663101298404164)

In [84]:
Evaluate(model, dev_loader)

/tmp/ipykernel_49/48991422.py:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(self.reviews[idx]),torch.tensor(float(self.sentiment[idx]=="positive"))


Accuracy: 0.8979 | Precision: 0.8832 | Recall: 0.9179


(0.8978666666666667, 0.8832098134423716, 0.9179282868525896)

In [85]:
Evaluate(model, test_loader)

/tmp/ipykernel_49/48991422.py:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  return torch.tensor(self.reviews[idx]),torch.tensor(float(self.sentiment[idx]=="positive"))


Accuracy: 0.8967 | Precision: 0.8818 | Recall: 0.9163


(0.8966666666666666, 0.8817645550141062, 0.9163113006396588)

In [87]:
def predict(text):
    model.eval()
    ids = torch.tensor(encode(tokenize(text)))          
    x = embedding(ids).unsqueeze(0).to(device)         
    with torch.no_grad():
        prob = torch.sigmoid(model(x)).item()         

    label = "positive" if prob > 0.5 else "negative"
    confidence = prob if prob > 0.5 else 1 - prob
    return label, confidence

In [110]:
predict("this movie has one of descent plot twist I have ever seen")

('positive', 0.8486993312835693)

In [111]:
predict("this movie has one of foolest plot twist I have ever seen")

('negative', 0.5320691168308258)